In [1]:
import os
import json
import scanpy as sc
import numpy as np
from vitessce.data_utils import (
    to_diamond,
    rgb_img_to_ome_zarr,
    optimize_adata,
    to_uint8
)

/usr/local/lib/python3.10/dist-packages/vitessce/__init__.py:42: UserWarning: Extra installs are necessary to use widgets: No module named 'anywidget'
  warn(f'Extra installs are necessary to use widgets: {e}')
/usr/local/lib/python3.10/dist-packages/vitessce/__init__.py:68: UserWarning: Extra installs are necessary to use exports: No module named 'starlette'
  warn(f'Extra installs are necessary to use exports: {e}')


In [2]:
# Load in the SRT anndata
all_samples = sc.read_h5ad("/zata/zippy/kresgeb/nmf_stuff/dlPFC/data/srt_with_nmf.h5ad")
all_samples

/usr/local/lib/python3.10/dist-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


AnnData object with n_obs × n_vars = 113927 × 28916
    obs: 'age', 'array_col', 'array_row', 'BayesSpace_harmony_02', 'BayesSpace_harmony_03', 'BayesSpace_harmony_04', 'BayesSpace_harmony_05', 'BayesSpace_harmony_06', 'BayesSpace_harmony_07', 'BayesSpace_harmony_08', 'BayesSpace_harmony_09', 'BayesSpace_harmony_10', 'BayesSpace_harmony_11', 'BayesSpace_harmony_12', 'BayesSpace_harmony_13', 'BayesSpace_harmony_14', 'BayesSpace_harmony_15', 'BayesSpace_harmony_16', 'BayesSpace_harmony_17', 'BayesSpace_harmony_18', 'BayesSpace_harmony_19', 'BayesSpace_harmony_20', 'BayesSpace_harmony_21', 'BayesSpace_harmony_22', 'BayesSpace_harmony_23', 'BayesSpace_harmony_24', 'BayesSpace_harmony_25', 'BayesSpace_harmony_26', 'BayesSpace_harmony_27', 'BayesSpace_harmony_28', 'BayesSpace_pca_02', 'BayesSpace_pca_03', 'BayesSpace_pca_04', 'BayesSpace_pca_05', 'BayesSpace_pca_06', 'BayesSpace_pca_07', 'BayesSpace_pca_08', 'BayesSpace_pca_09', 'BayesSpace_pca_10', 'BayesSpace_pca_11', 'BayesSpace_pca_12', 

In [3]:
# print the unique values of manual_layer_label across all samples
print(all_samples.obs["manual_layer_label"].unique())

[NaN, 'Layer 6', 'Layer 2', 'Layer 5', 'Layer 1', 'Layer 3', 'WM', 'Layer 4']
Categories (7, object): ['Layer 1', 'Layer 2', 'Layer 3', 'Layer 4', 'Layer 5', 'Layer 6', 'WM']


In [4]:
def split_by_sample(adata):
    """
    Split an AnnData object into a dictionary of subsets by sample_id.

    Args:
        adata (AnnData): The full AnnData object containing multiple samples with a 'sample_id' column in `.obs`.

    Returns:
        sample_dict (dict of str: Anndata): A dictionary mapping each unique sample_id (str) to a copied AnnData subset.

    Notes:
        Each subset is a copy, so modifying it does not affect the original AnnData.
        Progress is printed for each sample split.
    """
    sample_dict = {}
    sample_ids = adata.obs["sample_id"].unique()
    num_samples = len(sample_ids)
    for i, sample_id in enumerate(sample_ids, start=1):
        # Subset and copy
        sample_dict[sample_id] = adata[adata.obs["sample_id"] == sample_id].copy()
        # Drop these rows from adata
        adata = adata[adata.obs["sample_id"] != sample_id]
        print(f"Split {i}/{num_samples} samples ({i/num_samples:.2%} done)")

    return sample_dict

sample_dict = split_by_sample(all_samples)


Split 1/30 samples (3.33% done)
Split 2/30 samples (6.67% done)
Split 3/30 samples (10.00% done)
Split 4/30 samples (13.33% done)
Split 5/30 samples (16.67% done)
Split 6/30 samples (20.00% done)
Split 7/30 samples (23.33% done)
Split 8/30 samples (26.67% done)
Split 9/30 samples (30.00% done)
Split 10/30 samples (33.33% done)
Split 11/30 samples (36.67% done)
Split 12/30 samples (40.00% done)
Split 13/30 samples (43.33% done)
Split 14/30 samples (46.67% done)
Split 15/30 samples (50.00% done)
Split 16/30 samples (53.33% done)
Split 17/30 samples (56.67% done)
Split 18/30 samples (60.00% done)
Split 19/30 samples (63.33% done)
Split 20/30 samples (66.67% done)
Split 21/30 samples (70.00% done)
Split 22/30 samples (73.33% done)
Split 23/30 samples (76.67% done)
Split 24/30 samples (80.00% done)
Split 25/30 samples (83.33% done)
Split 26/30 samples (86.67% done)
Split 27/30 samples (90.00% done)
Split 28/30 samples (93.33% done)
Split 29/30 samples (96.67% done)
Split 30/30 samples (100.

In [5]:
def scale_spatial_coords(adata, sample_id):
    """
    Scale the spatial coordinates of a single AnnData object by its image scale factor.

    Args:
        adata (AnnData): AnnData object containing spatial coordinates in `.obsm["spatial"]`.
        sample_id (str): The sample_id corresponding to this AnnData, used to locate the scale factor 
                         in `.uns["spatial"][sample_id]["scalefactors"]["tissue_hires_scalef"]`.

    Notes:
        Modifies the `.obsm["spatial"]` field of `adata` in place.
    """
    # Get scale factor for this sample
    scale_factor = adata.uns["spatial"][sample_id]["scalefactors"]["tissue_hires_scalef"]
    # Overwrite with scaled coordinates
    adata.obsm["spatial"] = adata.obsm["spatial"] * scale_factor


In [6]:
def add_segmentations(adata, radius=7):
    """
    Add diamond-shaped segmentations to an AnnData object using spatial coordinates.

    Args:
        adata (AnnData): AnnData object with `.obsm["spatial"]` coordinates.
        radius (int): Radius for diamond shape.

    Notes:
        Modifies the `.obsm["segmentations"]` field of `adata` in place.
    """
    spatial = adata.obsm["spatial"]
    segmentations = [to_diamond(x, y, radius) for x, y in spatial]
    adata.obsm["segmentations"] = np.array(segmentations)


In [7]:
def write_ome_zarr_image(adata, sample_id, output_path):
    """
    Write the high-resolution RGB image from an AnnData object to OME-Zarr format.

    Args:
        adata (AnnData): AnnData object containing the high-resolution image at 
            `.uns["spatial"][sample_id]["images"]["hires"]`.
        sample_id (str): Key for the sample in `.uns["spatial"]`.
        output_path (str): Destination path for the OME-Zarr output (e.g., "image.ome.zarr").
        

    Raises:
        ValueError: If the high-resolution image is missing for the specified sample_id.
    """
    try:
        # Extract the RGB image from uns
        img_hires = adata.uns["spatial"][sample_id]["images"]["hires"]
    except KeyError as e:
        raise ValueError(f"Missing hires image for sample '{sample_id}'") from e

    # Convert from interleaved (H, W, C) to channel-first (C, H, W)
    img_arr = np.transpose(img_hires, (2, 0, 1))

    # Save image to OME-Zarr
    rgb_img_to_ome_zarr(
        img_arr,
        output_path,
        axes="cyx",              # color, y, x
        chunks=(1, 256, 256),    # default chunking
        img_name="H & E Image"   # name shown in Vitessce
    )


In [8]:
def make_vitessce_config(sample_id, base_output_dir, template_path, has_manual_layers = False):
    """
    Create a Vitessce config JSON for a given sample_id by 
    replacing `<<SAMPLE_NAME>>` in the template.

    Args:
        sample_id (str): Sample identifier to insert.
        base_output_dir (str): Path to the parent sample directory (contains 'data').
        template_path (str): Path to the template_config.json file.
        has_manual_layers (bool): Whether the sample has manually annotated layers to include.
    """
    # Load the template
    with open(template_path, "r") as f:
        data = json.load(f)

    # Replace <<SAMPLE_NAME>> with the actual sample name
    data_str = json.dumps(data)
    data_str = data_str.replace("<<SAMPLE_NAME>>", sample_id)
    data = json.loads(data_str)

    # Remove the manual layers obsSet if it does not exist in obs_cols
    # Note: This is only relevant for 2024, where the manual layers are not always present
    if not has_manual_layers:
        # Remove "Manually Annotated Layers" from obsSets
        datasets = data.get("datasets", [])
        for dataset in datasets:
            for file in dataset.get("files", []):
                options = file.get("options", {})
                obs_sets = options.get("obsSets", [])
                options["obsSets"] = [
                    entry for entry in obs_sets if entry["name"] != "Manually Annotated Layers"
                ]

    # Make configs directory (sibling to data)
    configs_dir = os.path.join(base_output_dir, "configs")
    os.makedirs(configs_dir, exist_ok=True)

    # Save output config
    output_path = os.path.join(configs_dir, f"{sample_id}_config.json")
    print(f"\tSaving Vitessce config to {output_path}")
    with open(output_path, "w") as f:
        json.dump(data, f, indent=2)
    
    return output_path

In [9]:
def prepare_obs_cols(adata):
    """
    Renames columns for BayesSpace into a format that is similar to the PsychSCREEN processing pipeline.
    Also determines if manual layer annotations are present for the sample.
    Removes spots without manual layer annotations if there are any WITH manual layers.

    Args:
        adata (AnnData): Input sample adata to prepare columns for (mutates)

    Returns:
        has_manual_layers (bool): Whether or not this AnnData has manual layer annotations
        adata (AnnData): The modified AnnData with updated obs columns.
    """
    has_manual_layers = False

    # Determines if the sample adata has manual layer information, and renames if necessary
    if "manual_layer_label" in adata.obs and adata.obs["manual_layer_label"].notna().any():
            #remove all spots that are NaN in manual_layer_label
            adata = adata[adata.obs["manual_layer_label"].notna()].copy()
            adata.obs["manual_layers"] = adata.obs["manual_layer_label"]
            print("Adata after filtering for manual layers:")
            print(adata)
            has_manual_layers = True

    # Remap BayesSpace harmony cluster columns to new names
    for k in [9, 16]:
        old_col = f"BayesSpace_harmony_{k:02d}"
        new_col = f"bayes_space_k={k}"
        if old_col in adata.obs:
            adata.obs[new_col] = adata.obs[old_col]
    print("Adata after preparing obs cols (inside):")
    print(adata)
    return has_manual_layers, adata


    

In [10]:
def make_nmf_adata(adata, sample_id, output_path, prefix="nmf"):
    """
    Create and save a new AnnData object containing NMF features 
    (obs['nmf1'] ... obs['nmf100']) as its expression matrix.

    Args:
        adata (AnnData): Input AnnData with nmf1...nmf100 in .obs.
        sample_id (str): Sample identifier.
        output_path (str): Path to save the resulting AnnData as zarr.
    """
    # Collect NMF columns
    nmf_cols = [col for col in adata.obs.columns if col.startswith(prefix)]
    nmf_matrix = adata.obs[nmf_cols].copy()

    # Replace NaNs with 0.0 and report
    nan_counts = nmf_matrix.isna().sum()
    total_nans = nan_counts.sum()
    if total_nans > 0:
        print(f"\t\t⚠️ Found {total_nans} NaNs across {nan_counts[nan_counts > 0].shape[0]} components.")
        print("\t\tPer-component NaN counts:")
        for idx, val in nan_counts[nan_counts > 0].items():
            print(f"\t\t\t{idx}: {val}")
        nmf_matrix = nmf_matrix.fillna(0.0)
        print("\t\t✅ Replaced all NaNs with 0.0")
    else:
        print("\t\tNo NaNs found in NMF features.")

    # Build new AnnData
    nmf_adata = sc.AnnData(
        X=nmf_matrix.to_numpy(),
        obs=adata.obs.copy(),
        obsm={k: adata.obsm[k] for k in ["spatial", "segmentations"] if k in adata.obsm},
        uns={"sample_id": sample_id}
    )
    nmf_adata.var_names = nmf_cols  # label the columns as nmf1...nmf100 (changes with prefix since they may not be called "nmf")
    
    has_manual_layers, adata = prepare_obs_cols(nmf_adata)
    
    # Optimize before saving
    optimized_nmf = optimize_adata(
        nmf_adata,
        # Need this in order to have violin plots (cannot pull from both datasets)
        obs_cols=["bayes_space_k=9", "bayes_space_k=16", "manual_layers"] if has_manual_layers else ["bayes_space_k=9", "bayes_space_k=16"],
        obsm_keys=["spatial", "segmentations"],
        optimize_X=True,
        to_dense_X=True,
    )

    # Save
    print(f"\tSaving NMF AnnData to {output_path}")
    optimized_nmf.write_zarr(output_path, chunks=[optimized_nmf.shape[0], 10])


In [11]:
output_dir = "/zata/public_html/users/kresgeb/dlpfc_nmf"
config_template_path = "/zata/zippy/kresgeb/nmf_stuff/dlPFC/resources/dlpfc_template_config.json"



num_samples = len(sample_dict)

for i, (sample_id, adata) in enumerate(sample_dict.items(), start=1):
    pct_done = (i / num_samples) * 100
    print(f"Processing {sample_id} (sample {i}/{num_samples}, {pct_done:.2f}% done)")

    print("\tApplying scalefactor...")
    scale_spatial_coords(adata, sample_id)

    print("\tAdding segmentations...")
    add_segmentations(adata)

    print("\tPreparing obs_cols...")
    has_manual_layers, adata = prepare_obs_cols(adata)

    print("Adata after preparing obs cols (outside):")
    print(adata)

    print(f"\tHas manual layers: {has_manual_layers}")

    print("\tOptimizing AnnData...")
    optimized_adata = optimize_adata(
        adata,
        obs_cols=["bayes_space_k=9", "bayes_space_k=16", "manual_layers"] if has_manual_layers else ["bayes_space_k=9", "bayes_space_k=16"],
        obsm_keys=["spatial", "segmentations"],
        layer_keys=["logcounts"],
        optimize_X=True,
        to_dense_X=True,
    )

    # Paths for saving
    sample_output_dir = os.path.join(output_dir, "data", sample_id)
    os.makedirs(sample_output_dir, exist_ok=True)

    # Save optimized AnnData
    optimized_adata_path = os.path.join(sample_output_dir, "data.h5ad.zarr")
    print(f"\tSaving AnnData to {optimized_adata_path}")
    optimized_adata.write_zarr(optimized_adata_path, chunks=[optimized_adata.shape[0], 10])

    # Create and save NMF AnnData
    paper_nmf_adata_path = os.path.join(sample_output_dir, "nmf_data.h5ad.zarr")
    print(f"\tCreating NMF AnnData...")
    make_nmf_adata(adata, sample_id, paper_nmf_adata_path)

    # Save the image (OME-Zarr)
    image_output_path = os.path.join(sample_output_dir, "image.ome.zarr")
    print(f"\tSaving image to {image_output_path}")
    write_ome_zarr_image(adata, sample_id, image_output_path)

    # Create and save config json
    print(f"\tCreating sample config...")
    make_vitessce_config(sample_id, output_dir, config_template_path, has_manual_layers)

Processing Br2743_ant (sample 1/30, 3.33% done)
	Applying scalefactor...
	Adding segmentations...
	Preparing obs_cols...
Adata after preparing obs cols (inside):
AnnData object with n_obs × n_vars = 4068 × 28916
    obs: 'age', 'array_col', 'array_row', 'BayesSpace_harmony_02', 'BayesSpace_harmony_03', 'BayesSpace_harmony_04', 'BayesSpace_harmony_05', 'BayesSpace_harmony_06', 'BayesSpace_harmony_07', 'BayesSpace_harmony_08', 'BayesSpace_harmony_09', 'BayesSpace_harmony_10', 'BayesSpace_harmony_11', 'BayesSpace_harmony_12', 'BayesSpace_harmony_13', 'BayesSpace_harmony_14', 'BayesSpace_harmony_15', 'BayesSpace_harmony_16', 'BayesSpace_harmony_17', 'BayesSpace_harmony_18', 'BayesSpace_harmony_19', 'BayesSpace_harmony_20', 'BayesSpace_harmony_21', 'BayesSpace_harmony_22', 'BayesSpace_harmony_23', 'BayesSpace_harmony_24', 'BayesSpace_harmony_25', 'BayesSpace_harmony_26', 'BayesSpace_harmony_27', 'BayesSpace_harmony_28', 'BayesSpace_pca_02', 'BayesSpace_pca_03', 'BayesSpace_pca_04', 'BayesSp

	Saving AnnData to /zata/public_html/users/kresgeb/dlpfc_nmf/data/Br2743_ant/data.h5ad.zarr
	Creating NMF AnnData...
		No NaNs found in NMF features.
Adata after preparing obs cols (inside):
AnnData object with n_obs × n_vars = 4068 × 80
    obs: 'age', 'array_col', 'array_row', 'BayesSpace_harmony_02', 'BayesSpace_harmony_03', 'BayesSpace_harmony_04', 'BayesSpace_harmony_05', 'BayesSpace_harmony_06', 'BayesSpace_harmony_07', 'BayesSpace_harmony_08', 'BayesSpace_harmony_09', 'BayesSpace_harmony_10', 'BayesSpace_harmony_11', 'BayesSpace_harmony_12', 'BayesSpace_harmony_13', 'BayesSpace_harmony_14', 'BayesSpace_harmony_15', 'BayesSpace_harmony_16', 'BayesSpace_harmony_17', 'BayesSpace_harmony_18', 'BayesSpace_harmony_19', 'BayesSpace_harmony_20', 'BayesSpace_harmony_21', 'BayesSpace_harmony_22', 'BayesSpace_harmony_23', 'BayesSpace_harmony_24', 'BayesSpace_harmony_25', 'BayesSpace_harmony_26', 'BayesSpace_harmony_27', 'BayesSpace_harmony_28', 'BayesSpace_pca_02', 'BayesSpace_pca_03', 'Ba

In [12]:
import pandas as pd

def make_sample_summary_with_links(
    sample_dict,
    output_csv="/zata/zippy/kresgeb/nmf_stuff/dlPFC/output/vitessce_spreadsheet.csv"
):
    """
    Generate a CSV summarizing samples with has_manual_layers, metadata, and Vitessce links.

    Args:
        sample_dict (dict): Dictionary mapping sample_id -> AnnData object.
        output_csv (str): Path to save the summary CSV.

    Returns:
        pd.DataFrame: DataFrame with sample_id, has_manual_layers, sex, age, position, subject, vitessce_link.
    """
    # Base URL for configs (already URL-encoded up to 'configs/')
    base_url = (
        "https://vitessce.io/#?edit=false&url="
        "https%3A%2F%2Fusers.wenglab.org%2Fkresgeb%2Fdlpfc_nmf%2Fconfigs%2F"
    )

    meta_cols = ["subject", "position","sex", "age"]
    records = []

    for sample_id, adata in sample_dict.items():
        has_manual_layers = "manual_layers" in adata.obs
        vitessce_link = f"{base_url}{sample_id}_config.json"

        record = {"sample_id": sample_id,
                  "has_manual_layers": has_manual_layers,
                  "vitessce_link": vitessce_link}

        # Check metadata columns
        for col in meta_cols:
            if col in adata.obs:
                unique_vals = adata.obs[col].dropna().unique()
                if len(unique_vals) == 1:
                    record[col] = unique_vals[0]
                else:
                    record[col] = None
                    print(f"⚠️ Sample {sample_id} has multiple unique values for '{col}': {unique_vals}")
            else:
                record[col] = None

        records.append(record)

    df = pd.DataFrame(records)
    df.to_csv(output_csv, index=False)
    print(f"✅ Saved sample summary CSV with Vitessce links to {output_csv}")
    return df

# Example usage
summary_df = make_sample_summary_with_links(sample_dict)
summary_df.head()

✅ Saved sample summary CSV with Vitessce links to /zata/zippy/kresgeb/nmf_stuff/dlPFC/output/vitessce_spreadsheet.csv


,sample_id,has_manual_layers,vitessce_link,subject,position,sex,age
0,Br2743_ant,False,https://vitessce.io/#?edit=false&url=https%3A%...,Br2743,anterior,M,61.54
1,Br2743_mid,False,https://vitessce.io/#?edit=false&url=https%3A%...,Br2743,middle,M,61.54
2,Br2743_post,False,https://vitessce.io/#?edit=false&url=https%3A%...,Br2743,posterior,M,61.54
3,Br3942_ant,False,https://vitessce.io/#?edit=false&url=https%3A%...,Br3942,anterior,M,47.53
4,Br3942_mid,False,https://vitessce.io/#?edit=false&url=https%3A%...,Br3942,middle,M,47.53
